## Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots “talk” because they are *vocal learners*—the only birds (besides humans and a few other species) that can imitate sounds from their environment.  What they call “talking” is really a sophisticated form of mimicry that serves several purposes in the wild and in captivity.\n\n---\n\n## 1.  The biology behind the mimicry\n\n| Feature | What it means for parrots | Why it matters |\n|---------|---------------------------|----------------|\n| **Vocal learning** | Parrots can modify the sounds they hear and produce new combinations. | Enables them to copy human speech, other bird calls, and even environmental noises. |\n| **Large syrinx (voice box)** | Parrots have a highly flexible syrinx with many muscles. | Allows fine‑tuned control of pitch, volume, and rhythm. |\n| **Neural circuitry** | They possess a large portion of their brain devoted to vocal control, similar to humans. | Supports the complex processing needed for mimicry. |\n| **Social brain** | Parrots l

In [8]:
from langchain.tools import tool
#I defined my tool here -> "If you ever need weather information, there's a tool called get_weather that accepts a location."
@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

#we bind it
model_with_tools=model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to call get_weather function.', 'tool_calls': [{'id': 'fc_8631315f-4b92-4c37-b8e7-f71fbde90953', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 127, 'total_tokens': 159, 'completion_time': 0.035207741, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.007246079, 'prompt_tokens_details': None, 'queue_time': 0.320246597, 'total_time': 0.04245382}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_63473661d7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fa9c0-ed62-7250-8907-ce2a098f002c-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_8631315f-4b92-4c37-b8e7-f71fbde90953', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 127, 'output_tokens'

In [6]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in UP?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The current weather in Uttar Pradesh is sunny. If you need more details (temperature, humidity, forecast for the week, etc.), just let me know!
